In [1]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [2]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [3]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [4]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

👉 加载主训练数据
  原始训练样本数: 7973
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条
  → 正在增强 Density 数据，共 787 条


[19:54:06] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[19:54:06] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[19:54:06] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[19:54:06] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[19:54:06] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[19:54:06] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[19:54:06] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[19:54:06] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[19:54:06] SMILES Parse 

cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081
👉 划分 train / validation / test
  划分结果: train=8064, val=1008, test=1009
Loaded: 8064 1008 1009


In [6]:
from train_stage1 import (setup_stage1_data, create_stage1_model, 
                        optimize_stage1, train_final_stage1_model)

/usr/local/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
print(train_df.head())

             id                                             SMILES  Tg  \
0  2.026603e+09    *Oc1ccc(CC2(Cc3ccc(*)cc3)c3ccccc3-c3ccccc32)cc1 NaN   
1  1.189403e+09         *CC(O)COc1ccc(C(C)CC(C)(C)c2ccc(O*)cc2)cc1 NaN   
2  1.686537e+09  *C(=O)c1ccc2c(c1)C(=O)N(c1c(C)cc(C(c3cc(C)c(N4... NaN   
3  2.632934e+08  *Oc1ccc2ccc(Oc3ccc(C(=Nc4ccc(N=C(c5ccccc5)c5cc... NaN   
4  1.280165e+09                    *Nc1ccc(-c2ccc(N*)c(OC)c2)cc1OC NaN   

        FFV  Tc  Density  Rg  
0  0.386736 NaN      NaN NaN  
1  0.354235 NaN      NaN NaN  
2  0.430396 NaN      NaN NaN  
3  0.381740 NaN      NaN NaN  
4  0.334115 NaN      NaN NaN  


In [ ]:
# 运行Optuna优化
study = optimize_stage1(
    train_df,
    study_name="my_polymer_study",
    n_trials=50,  # 可以根据需要调整
    patience=15
)

# 查看最佳参数
print("Best trial:")
print(" Value: ", study.best_trial.value)
print(" Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条


[I 2025-08-01 19:54:34,886] A new study created in RDB with name: my_polymer_study


In [11]:
# 6) 用 lambda／partial 把额外参数绑进去
study1.optimize(
    lambda trial: objective_stage1(trial, nodeedge_loader, device),
    n_trials=20,
    show_progress_bar=True
)

Best trial: 1. Best value: 0.392674:   5%|▌         | 1/20 [10:28<3:19:01, 628.49s/it]

[I 2025-08-01 18:07:33,276] Trial 1 finished with value: 0.3926738825818849 and parameters: {'lr': 2.059267556393373e-05, 'hidden_dim': 128, 'num_edge_layers': 2}. Best is trial 1 with value: 0.3926738825818849.


Best trial: 2. Best value: 0.315944:  10%|█         | 2/20 [17:04<2:27:32, 491.81s/it]

[I 2025-08-01 18:14:09,415] Trial 2 finished with value: 0.3159444455116514 and parameters: {'lr': 0.00011164166128266972, 'hidden_dim': 256, 'num_edge_layers': 4}. Best is trial 2 with value: 0.3159444455116514.


Best trial: 3. Best value: 0.204353:  15%|█▌        | 3/20 [23:09<2:02:55, 433.87s/it]

...

[I 2025-08-01 18:37:22,462] Trial 18 finished with value: 0.20874204729048032 and parameters: {'lr': 0.0035719408754786713, 'hidden_dim': 64, 'num_edge_layers': 3}. Best is trial 3 with value: 0.20435308499468696.


Best trial: 3. Best value: 0.204353:  95%|█████████▌| 19/20 [40:22<01:13, 73.26s/it] 

[I 2025-08-01 18:37:27,042] Trial 19 pruned. 


Best trial: 3. Best value: 0.204353: 100%|██████████| 20/20 [40:26<00:00, 121.32s/it]

[I 2025-08-01 18:37:31,210] Trial 20 pruned. 


In [12]:
save_stage1_artifacts(study1)

✅ Saved best params & encoder to `stage1_artifacts` and cleaned up `tmp_stage1`.


In [ ]:
# 加载最佳参数
best_params = torch.load("stage1_artifacts/stage1_best_params_trialX.pt")

# 创建模型
encoder, model = create_stage1_model(best_params, device)

# 加载预训练权重（如果需要）
encoder.load_state_dict(torch.load("stage1_artifacts/stage1_encoder_best_trialX.pt"))

# 继续训练或用于下游任务
optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])